# Internal Confidence: Epistemic Uncertainty in the Residual Stream

This notebook demonstrates IRTK's `internal_confidence` module:

1. **Confidence direction** – linear direction predicting output confidence
2. **Internal vs output gap** – when model 'knows' but hasn't committed
3. **Accumulation profile** – how confidence builds through layers
4. **Uncertainty decomposition** – PCA of uncertainty-related variance
5. **Self-consistency probe** – stability across paraphrases

## Why This Matters

Internal confidence analysis finds directions in activation space that predict how confident the model is in its output, even when that confidence doesn't match the output probabilities. Gaps between internal confidence and output calibration reveal when models 'know they don't know.'

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax, jax.numpy as jnp, numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.internal_confidence import (
    confidence_direction, internal_vs_output_confidence_gap,
    confidence_accumulation_profile, uncertainty_decomposition,
    self_consistency_probe,
)

In [ ]:
cfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = [jax.random.normal((key := jax.random.split(key)[1]), l.shape) * 0.1
              if isinstance(l, jnp.ndarray) and l.dtype == jnp.float32 else l for l in leaves]
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 1, 2, 3, 4, 5])

In [ ]:
conf = confidence_direction(model, tokens, correct_token=5, hook_name='blocks.0.hook_resid_post')
print(f"Confidence score: {conf['confidence_score']:.4f}")
print(f"Output prob for token 5: {conf['output_prob']:.4f}")
print(f"Direction norm: {conf['direction_norm']:.4f}")

In [ ]:
gap = internal_vs_output_confidence_gap(model, tokens, correct_token=5)
print(f"Internal confidence per layer: {gap['layer_internal_conf'].round(4)}")
print(f"Output prob: {gap['output_prob']:.4f}")
print(f"Gaps: {gap['gaps'].round(4)}")
print(f"Max gap at layer: {gap['max_gap_layer']}")

In [ ]:
seqs = [jnp.array([0,1,2,3,4,5]), jnp.array([10,11,12,13,14,15])]
profile = confidence_accumulation_profile(model, seqs, [5, 15])
print(f"Mean confidence: {profile['mean_confidence'].round(4)}")
print(f"Std: {profile['std_confidence'].round(4)}")
print(f"Monotonicity: {profile['monotonicity']:.4f}")

In [ ]:
seqs = [jnp.array([0,1,2,3]), jnp.array([5,6,7,8]), jnp.array([10,11,12,13])]
decomp = uncertainty_decomposition(model, seqs, 'blocks.0.hook_resid_post', top_k=3)
print(f"Explained variance: {decomp['explained_variance'].round(4)}")
print(f"Entropy correlations: {decomp['entropy_correlations'].round(4)}")
print(f"Total variance: {decomp['total_variance']:.4f}")

In [ ]:
consist = self_consistency_probe(model, jnp.array([0,1,2,3]), jnp.array([0,1,2,4]),
                               'blocks.1.hook_resid_post')
print(f"Cosine similarity: {consist['cosine_similarity']:.4f}")
print(f"Output agreement: {consist['output_agreement']}")
print(f"KL divergence: {consist['kl_divergence']:.4f}")
print(f"Consistency score: {consist['consistency_score']:.4f}")